# ML2. Обучение с учителем: линейные модели

Цель работы — реализовать линейную регрессию и регуляризованные линейные модели, сравнить их с реализациями `sklearn`, исследовать нормализацию и переобучение на полиномиальных признаках.

## 1. Теоретические вопросы

### 1.1. Аналитическое решение линейной регрессии

Для матрицы признаков $X \in \mathbb{R}^{n \times d}$ и вектора целей $y \in \mathbb{R}^n$ минимизируется

$$L(\theta)=\|X\theta-y\|_2^2.$$

Условие нулевого градиента:

$$\nabla_\theta L=2X^T(X\theta-y)=0,$$

откуда нормальное уравнение и его решение:

$$X^TX\theta=X^Ty, \qquad \hat\theta=(X^TX)^{-1}X^Ty.$$

Если $X^TX$ вырождена, используется псевдообратная матрица: $\hat\theta=X^+y$. Свободный член можно учесть, добавив к $X$ столбец единиц.

### 1.2. Что меняют L1- и L2-регуляризации

В функцию потерь добавляется штраф:

$$\min_\theta \sum_{i=1}^{n}L(f(x_i,\theta),y_i)+\lambda R(\theta).$$

- L2 (Ridge): $R(\theta)=\|\theta\|_2^2$. Для коэффициентов без штрафа на свободный член аналитическое решение имеет вид $\hat\theta=(X^TX+\lambda I)^{-1}X^Ty$. Коэффициенты плавно уменьшаются, решение становится устойчивее при мультиколлинеарности.
- L1 (Lasso): $R(\theta)=\|\theta\|_1$. В общем случае закрытого аналитического решения нет из-за недифференцируемости модуля в нуле; применяются субградиентные или координатные методы.
- ElasticNet объединяет оба штрафа: $\lambda\bigl(\rho\|\theta\|_1+\frac{1-\rho}{2}\|\theta\|_2^2\bigr)$.

### 1.3. Почему L1 используется для отбора признаков

Каждый ненулевой коэффициент увеличивает L1-штраф. Оператор soft-thresholding уменьшает модуль коэффициента на фиксированный порог и обнуляет его, если модуль меньше порога. Поэтому слабые признаки получают коэффициент ровно 0, решение становится разреженным, а признаки с нулевыми весами можно исключить.

### 1.4. Нелинейные зависимости в линейных моделях

Линейной остаётся зависимость от параметров, но исходные признаки заменяются нелинейными преобразованиями $\phi(x)$: степенями и взаимодействиями, `log`, `exp`, `sin` и другими функциями. Тогда модель $f(x)=w^T\phi(x)+b$ способна описывать нелинейную зависимость от исходного $x$.

## 2. Импорт библиотек и подготовка данных

In [1]:
from collections import Counter
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler, PolynomialFeatures

warnings.filterwarnings("ignore", category=ConvergenceWarning)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 100)
RANDOM_STATE = 42

In [2]:
cwd = Path.cwd().resolve()
project_root = cwd.parent if cwd.name == "src" else cwd
data_dir = project_root / "src" / "data"
train_path = data_dir / "train.json"
test_path = data_dir / "test.json"

# Оба файла читаются по заданию. Для регрессии продолжаем работу с train.json
# из ML1: competition test.json не является test-частью предыдущего проекта.
train = pd.read_json(train_path)
kaggle_test = pd.read_json(test_path)
print(f"Исходный train.json: {train.shape}")
print(f"Исходный Kaggle test.json: {kaggle_test.shape}")
display(train[["bathrooms", "bedrooms", "features", "price"]].head())


Исходный train.json: (49352, 15)
Исходный Kaggle test.json: (74659, 14)


,bathrooms,bedrooms,features,price
4,1.0,1,"[Dining Room, Pre-War, Laundry in Building, Di...",2400
6,1.0,2,"[Doorman, Elevator, Laundry in Building, Dishw...",3800
9,1.0,2,"[Doorman, Elevator, Laundry in Building, Laund...",3495
10,1.5,3,[],3000
15,1.0,0,"[Doorman, Elevator, Fitness Center, Laundry in...",2795


Предобработка строго продолжает предыдущий проект ML1. В качестве единого исходного `dataset` используется `train.json`: из него удаляются строки с `price` вне открытого интервала между 1-м и 99-м процентилями, после чего очищенный набор воспроизводимо делится на `train` и `test` в отношении 75/25 (`random_state=1`).

Файл соревнования Kaggle `test.json` прочитан по заданию, но не подставляется вместо контрольной части ML1: такое смешение двух исходных файлов меняет распределение данных и делает метрики несопоставимыми с чек-листом.


In [3]:
price_low, price_high = train["price"].quantile([0.01, 0.99])
clean_data = train[
    (train["price"] > price_low) & (train["price"] < price_high)
].copy()

train_data, test_data = train_test_split(
    clean_data,
    test_size=0.25,
    random_state=1,
)

print(f"Границы price: ({price_low}, {price_high})")
print(f"После удаления хвостов: {clean_data.shape}")
print(f"Train: {train_data.shape}; test: {test_data.shape}")


Границы price: (1475.0, 13000.0)
После удаления хвостов: (48343, 15)
Train: (36257, 15); test: (12086, 15)


## 3. Дополнительные признаки из столбца `features`

In [4]:
def clean_feature_cell(values):
    # Удаляет [, ], кавычки и пробелы, затем возвращает элементы списка.
    cleaned = str(values)
    for symbol in ("[", "]", "'", '"', " "):
        cleaned = cleaned.replace(symbol, "")
    return cleaned.split(",")


train_data["features_clean"] = train_data["features"].apply(clean_feature_cell)
test_data["features_clean"] = test_data["features"].apply(clean_feature_cell)

all_train_features = []
for _, row in train_data.iterrows():
    all_train_features.extend(row["features_clean"])

cnt = Counter(all_train_features)
print(f"Уникальных значений в train: {len(cnt)}")
print("21 наиболее частое значение (включая пустую строку):")
cnt.most_common(21)


Уникальных значений в train: 1280
21 наиболее частое значение (включая пустую строку):


[('Elevator', 18981),
 ('HardwoodFloors', 17336),
 ('CatsAllowed', 17289),
 ('DogsAllowed', 16191),
 ('Doorman', 15326),
 ('Dishwasher', 14943),
 ('NoFee', 13269),
 ('LaundryinBuilding', 12013),
 ('FitnessCenter', 9690),
 ('Pre-War', 6769),
 ('LaundryinUnit', 6350),
 ('RoofDeck', 4767),
 ('OutdoorSpace', 3816),
 ('DiningRoom', 3658),
 ('HighSpeedInternet', 3130),
 ('', 2318),
 ('Balcony', 2187),
 ('SwimmingPool', 1974),
 ('LaundryInBuilding', 1906),
 ('NewConstruction', 1848),
 ('Terrace', 1629)]

In [5]:
calculated_top_20 = [name for name, _ in cnt.most_common(21) if name][:20]
expected_top_20 = [
    "Elevator", "HardwoodFloors", "CatsAllowed", "DogsAllowed", "Doorman",
    "Dishwasher", "NoFee", "LaundryinBuilding", "FitnessCenter", "Pre-War",
    "LaundryinUnit", "RoofDeck", "OutdoorSpace", "DiningRoom",
    "HighSpeedInternet", "Balcony", "SwimmingPool", "LaundryInBuilding",
    "NewConstruction", "Terrace",
]

assert set(calculated_top_20) == set(expected_top_20)
top_20_features = calculated_top_20


def add_binary_feature_columns(frame):
    result = frame.copy()
    result["features_clean"] = result["features"].apply(clean_feature_cell)
    for feature in top_20_features:
        result[feature] = result["features_clean"].apply(
            lambda values, f=feature: int(f in values)
        )
    return result


train_data = add_binary_feature_columns(train_data)
test_data = add_binary_feature_columns(test_data)

# Для бонусного сравнения test фиксируется после обязательной основной
# предобработки. Дополнительное отсечение выполняется только в train.
bonus_train_raw = train_data.copy()
bonus_test_raw = test_data.copy()

bonus_low, bonus_high = bonus_train_raw["price"].quantile([0.01, 0.99])
bonus_train_clean = bonus_train_raw[
    (bonus_train_raw["price"] > bonus_low)
    & (bonus_train_raw["price"] < bonus_high)
].copy()

feature_list = top_20_features + ["bathrooms", "bedrooms"]
assert len(feature_list) == 22

X_train = train_data[feature_list].to_numpy(dtype=float)
X_test = test_data[feature_list].to_numpy(dtype=float)
y_train = train_data["price"].to_numpy(dtype=float)
y_test = test_data["price"].to_numpy(dtype=float)

print(f"Итоговый список из {len(feature_list)} признаков:")
print(feature_list)
print(f"X_train: {X_train.shape}; X_test: {X_test.shape}")


Итоговый список из 22 признаков:
['Elevator', 'HardwoodFloors', 'CatsAllowed', 'DogsAllowed', 'Doorman', 'Dishwasher', 'NoFee', 'LaundryinBuilding', 'FitnessCenter', 'Pre-War', 'LaundryinUnit', 'RoofDeck', 'OutdoorSpace', 'DiningRoom', 'HighSpeedInternet', 'Balcony', 'SwimmingPool', 'LaundryInBuilding', 'NewConstruction', 'Terrace', 'bathrooms', 'bedrooms']
X_train: (36257, 22); X_test: (12086, 22)


## 4. Собственные реализации моделей и метрики

Детерминированная модель при одинаковых данных и параметрах каждый раз возвращает одинаковый результат. В собственной SGD-реализации это обеспечивается фиксированным `random_state`, одинаковой инициализацией и воспроизводимым порядком перемешивания. Обучение идёт мини-пакетами, то есть используется стохастическая оценка градиента.

Коэффициент детерминации:

$$R^2=1-\frac{\sum_i(y_i-\hat y_i)^2}{\sum_i(y_i-\bar y)^2}.$$

In [6]:
def r2_custom(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return 1.0 - np.sum((y_true - y_pred) ** 2) / np.sum((y_true - y_true.mean()) ** 2)


class CustomSGDLinearRegression:
    def __init__(self, learning_rate=0.02, epochs=250, batch_size=256, random_state=42):
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size
        self.random_state = random_state

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)
        self.x_mean_ = X.mean(axis=0)
        self.x_scale_ = X.std(axis=0)
        self.x_scale_[self.x_scale_ == 0] = 1.0
        X_scaled = (X - self.x_mean_) / self.x_scale_

        self.y_mean_ = y.mean()
        y_centered = y - self.y_mean_
        self.coef_scaled_ = np.zeros(X.shape[1], dtype=float)
        self.intercept_scaled_ = 0.0
        rng = np.random.default_rng(self.random_state)

        for epoch in range(self.epochs):
            eta = self.learning_rate / (1.0 + 0.01 * epoch)
            indices = rng.permutation(len(X_scaled))
            for batch in np.array_split(indices, int(np.ceil(len(indices) / self.batch_size))):
                error = X_scaled[batch] @ self.coef_scaled_ + self.intercept_scaled_ - y_centered[batch]
                gradient_w = X_scaled[batch].T @ error / len(batch)
                gradient_b = error.mean()
                self.coef_scaled_ -= eta * gradient_w
                self.intercept_scaled_ -= eta * gradient_b

        self.coef_ = self.coef_scaled_ / self.x_scale_
        self.intercept_ = (
            self.y_mean_ + self.intercept_scaled_ - self.x_mean_ @ self.coef_
        )
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        return X @ self.coef_ + self.intercept_


class CustomRidgeRegression:
    def __init__(self, alpha=1.0):
        self.alpha = alpha

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)
        self.x_mean_ = X.mean(axis=0)
        self.y_mean_ = y.mean()
        X_centered = X - self.x_mean_
        y_centered = y - self.y_mean_
        matrix = X_centered.T @ X_centered + self.alpha * np.eye(X.shape[1])
        self.coef_ = np.linalg.pinv(matrix) @ X_centered.T @ y_centered
        self.intercept_ = self.y_mean_ - self.x_mean_ @ self.coef_
        return self

    def predict(self, X):
        return np.asarray(X, dtype=float) @ self.coef_ + self.intercept_


class CustomElasticNetRegression:
    # Циклический координатный спуск для L1 и ElasticNet.
    def __init__(self, alpha=1.0, l1_ratio=0.5, max_iter=2000, tol=1e-7):
        self.alpha = alpha
        self.l1_ratio = l1_ratio
        self.max_iter = max_iter
        self.tol = tol

    @staticmethod
    def _soft_threshold(value, threshold):
        return np.sign(value) * max(abs(value) - threshold, 0.0)

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)
        self.x_mean_ = X.mean(axis=0)
        self.y_mean_ = y.mean()
        X_centered = X - self.x_mean_
        y_centered = y - self.y_mean_
        n_samples, n_features = X_centered.shape

        weights = np.zeros(n_features, dtype=float)
        residual = y_centered.copy()
        squared_norms = np.sum(X_centered ** 2, axis=0)
        l1_penalty = n_samples * self.alpha * self.l1_ratio
        l2_penalty = n_samples * self.alpha * (1.0 - self.l1_ratio)

        for iteration in range(self.max_iter):
            max_change = 0.0
            for j in range(n_features):
                residual += X_centered[:, j] * weights[j]
                rho = X_centered[:, j] @ residual
                if squared_norms[j] == 0:
                    new_weight = 0.0
                else:
                    new_weight = self._soft_threshold(rho, l1_penalty) / (
                        squared_norms[j] + l2_penalty
                    )
                residual -= X_centered[:, j] * new_weight
                max_change = max(max_change, abs(new_weight - weights[j]))
                weights[j] = new_weight
            if max_change < self.tol:
                break

        self.n_iter_ = iteration + 1
        self.coef_ = weights
        self.intercept_ = self.y_mean_ - self.x_mean_ @ self.coef_
        return self

    def predict(self, X):
        return np.asarray(X, dtype=float) @ self.coef_ + self.intercept_


class CustomLassoRegression(CustomElasticNetRegression):
    def __init__(self, alpha=1.0, max_iter=2000, tol=1e-7):
        super().__init__(alpha=alpha, l1_ratio=1.0, max_iter=max_iter, tol=tol)

In [7]:
def calculate_metrics(y_true, prediction):
    return {
        "MAE": mean_absolute_error(y_true, prediction),
        "RMSE": np.sqrt(mean_squared_error(y_true, prediction)),
        "R2": r2_custom(y_true, prediction),
    }


def evaluate_pair(name, custom_model, sklearn_model, X_tr, X_te):
    rows = []
    for implementation, model in (("custom", custom_model), ("sklearn", sklearn_model)):
        model.fit(X_tr, y_train)
        train_prediction = model.predict(X_tr)
        test_prediction = model.predict(X_te)
        train_metrics = calculate_metrics(y_train, train_prediction)
        test_metrics = calculate_metrics(y_test, test_prediction)
        rows.append({
            "model": name,
            "implementation": implementation,
            "train_MAE": train_metrics["MAE"],
            "test_MAE": test_metrics["MAE"],
            "train_RMSE": train_metrics["RMSE"],
            "test_RMSE": test_metrics["RMSE"],
            "train_R2": train_metrics["R2"],
            "test_R2": test_metrics["R2"],
        })
    return rows


default_comparison_rows = []
default_comparison_rows += evaluate_pair(
    "LinearRegression",
    CustomSGDLinearRegression(random_state=RANDOM_STATE),
    LinearRegression(),
    X_train,
    X_test,
)
default_comparison_rows += evaluate_pair(
    "Ridge",
    CustomRidgeRegression(alpha=1.0),
    Ridge(alpha=1.0),
    X_train,
    X_test,
)
default_comparison_rows += evaluate_pair(
    "Lasso",
    CustomLassoRegression(alpha=1.0),
    Lasso(alpha=1.0, max_iter=10000),
    X_train,
    X_test,
)
default_comparison_rows += evaluate_pair(
    "ElasticNet",
    CustomElasticNetRegression(alpha=1.0, l1_ratio=0.5),
    ElasticNet(alpha=1.0, l1_ratio=0.5, max_iter=10000),
    X_train,
    X_test,
)

default_comparison = pd.DataFrame(default_comparison_rows)
display(default_comparison)

,model,implementation,train_MAE,test_MAE,train_RMSE,test_RMSE,train_R2,test_R2
0,LinearRegression,custom,706.024534,717.475660,1023.192388,1040.012607,0.576819,0.589411
1,LinearRegression,sklearn,705.816155,717.318394,1023.172906,1039.977925,0.576835,0.589438
2,Ridge,custom,705.812250,717.316223,1023.172915,1039.981056,0.576835,0.589436
3,Ridge,sklearn,705.812250,717.316223,1023.172915,1039.981056,0.576835,0.589436
4,Lasso,custom,705.418130,717.141108,1023.371560,1040.305448,0.576671,0.589180
5,Lasso,sklearn,705.419664,717.142644,1023.372627,1040.306704,0.576670,0.589179
6,ElasticNet,custom,799.790866,821.964236,1174.285496,1207.941571,0.442611,0.446112
7,ElasticNet,sklearn,799.790808,821.964181,1174.285421,1207.941502,0.442611,0.446112


In [8]:
comparison = default_comparison.pivot(index="model", columns="implementation")
for metric in ("test_MAE", "test_RMSE"):
    comparison[(metric, "relative_difference")] = (
        (comparison[(metric, "custom")] - comparison[(metric, "sklearn")]).abs()
        / comparison[(metric, "sklearn")]
    )

print("Максимальная относительная разница MAE:", comparison[("test_MAE", "relative_difference")].max())
print("Максимальная относительная разница RMSE:", comparison[("test_RMSE", "relative_difference")].max())
print("Минимальный test R2 собственной реализации:", default_comparison.query("implementation == 'custom'")["test_R2"].min())
assert comparison[("test_MAE", "relative_difference")].max() < 0.10
assert comparison[("test_RMSE", "relative_difference")].max() < 0.10
assert default_comparison.query("implementation == 'custom'")["test_R2"].min() > 0.40

deterministic_a = CustomSGDLinearRegression(epochs=10, random_state=RANDOM_STATE).fit(
    X_train[:2048], y_train[:2048]
)
deterministic_b = CustomSGDLinearRegression(epochs=10, random_state=RANDOM_STATE).fit(
    X_train[:2048], y_train[:2048]
)
assert np.array_equal(
    deterministic_a.predict(X_test[:100]), deterministic_b.predict(X_test[:100])
)
assert np.isclose(r2_custom(y_test, deterministic_a.predict(X_test)),
                  r2_score(y_test, deterministic_a.predict(X_test)))
print("Повторный запуск SGD с тем же random_state полностью воспроизводим: True")

Максимальная относительная разница MAE: 0.00021924207697104775
Максимальная относительная разница RMSE: 3.3349035331338063e-05
Минимальный test R2 собственной реализации: 0.4461118603942643
Повторный запуск SGD с тем же random_state полностью воспроизводим: True


## 5. Нормализация признаков

Нормализация обязательна или особенно важна для методов, зависящих от расстояний и масштаба: градиентного спуска, KNN, SVM, PCA, нейронных сетей, а также линейных моделей с L1/L2-регуляризацией. Без неё признак с большим числовым диапазоном непропорционально влияет на расстояние, градиент или штраф.

Масштабирование обычно не обязательно для деревьев решений и их ансамблей, потому что разбиения зависят от порядка значений, а не от масштаба. Для обычной линейной регрессии без регуляризации изменение масштаба меняет коэффициенты, но не пространство прогнозов. Для one-hot признаков масштабирование также обычно не требуется.

MinMaxScaler:

$$x' = \frac{x-x_{\min}}{x_{\max}-x_{\min}}.$$

StandardScaler:

$$x' = \frac{x-\mu}{\sigma}.$$

Оба scaler обучаются только на train, затем без повторного `fit` применяются к train и test, чтобы не допустить утечку данных.

In [9]:
class CustomMinMaxScaler:
    def fit(self, X):
        X = np.asarray(X, dtype=float)
        self.min_ = X.min(axis=0)
        self.range_ = X.max(axis=0) - self.min_
        self.range_[self.range_ == 0] = 1.0
        return self

    def transform(self, X):
        return (np.asarray(X, dtype=float) - self.min_) / self.range_

    def fit_transform(self, X):
        return self.fit(X).transform(X)


class CustomStandardScaler:
    def fit(self, X):
        X = np.asarray(X, dtype=float)
        self.mean_ = X.mean(axis=0)
        self.scale_ = X.std(axis=0, ddof=0)
        self.scale_[self.scale_ == 0] = 1.0
        return self

    def transform(self, X):
        return (np.asarray(X, dtype=float) - self.mean_) / self.scale_

    def fit_transform(self, X):
        return self.fit(X).transform(X)


custom_minmax = CustomMinMaxScaler().fit(X_train)
sklearn_minmax = MinMaxScaler().fit(X_train)
custom_standard = CustomStandardScaler().fit(X_train)
sklearn_standard = StandardScaler().fit(X_train)

X_train_minmax = sklearn_minmax.transform(X_train)
X_test_minmax = sklearn_minmax.transform(X_test)
X_train_standard = sklearn_standard.transform(X_train)
X_test_standard = sklearn_standard.transform(X_test)

print("MinMax train совпадает:", np.allclose(custom_minmax.transform(X_train), X_train_minmax))
print("MinMax test совпадает:", np.allclose(custom_minmax.transform(X_test), X_test_minmax))
print("Standard train совпадает:", np.allclose(custom_standard.transform(X_train), X_train_standard))
print("Standard test совпадает:", np.allclose(custom_standard.transform(X_test), X_test_standard))

assert np.allclose(custom_minmax.transform(X_train), X_train_minmax)
assert np.allclose(custom_minmax.transform(X_test), X_test_minmax)
assert np.allclose(custom_standard.transform(X_train), X_train_standard)
assert np.allclose(custom_standard.transform(X_test), X_test_standard)

MinMax train совпадает: True
MinMax test совпадает: True
Standard train совпадает: True
Standard test совпадает: True


## 6. Все модели на исходных и нормализованных данных

In [10]:
def model_factories(custom=False):
    if custom:
        return {
            "Linear": lambda: CustomSGDLinearRegression(random_state=RANDOM_STATE),
            "Ridge": lambda: CustomRidgeRegression(alpha=1.0),
            "Lasso": lambda: CustomLassoRegression(alpha=1.0),
            "ElasticNet": lambda: CustomElasticNetRegression(alpha=1.0, l1_ratio=0.5),
        }
    return {
        "Linear": lambda: LinearRegression(),
        "Ridge": lambda: Ridge(alpha=1.0),
        "Lasso": lambda: Lasso(alpha=1.0, max_iter=10000),
        "ElasticNet": lambda: ElasticNet(alpha=1.0, l1_ratio=0.5, max_iter=10000),
    }


def evaluate_model_set(factories, datasets, implementation):
    rows = []
    fitted = {}
    for dataset_name, (X_tr, X_te) in datasets.items():
        for model_name, factory in factories.items():
            model = factory().fit(X_tr, y_train)
            fitted[(dataset_name, model_name)] = model
            train_metrics = calculate_metrics(y_train, model.predict(X_tr))
            test_metrics = calculate_metrics(y_test, model.predict(X_te))
            rows.append({
                "model": f"{model_name} {dataset_name}",
                "implementation": implementation,
                "train_MAE": train_metrics["MAE"],
                "test_MAE": test_metrics["MAE"],
                "train_RMSE": train_metrics["RMSE"],
                "test_RMSE": test_metrics["RMSE"],
                "train_R2": train_metrics["R2"],
                "test_R2": test_metrics["R2"],
            })
    return pd.DataFrame(rows), fitted


datasets = {
    "default": (X_train, X_test),
    "MinMaxScaler": (X_train_minmax, X_test_minmax),
    "StandardScaler": (X_train_standard, X_test_standard),
}

custom_all_metrics, custom_fitted = evaluate_model_set(model_factories(custom=True), datasets, "custom")
sklearn_all_metrics, sklearn_fitted = evaluate_model_set(model_factories(custom=False), datasets, "sklearn")

display(pd.concat([custom_all_metrics, sklearn_all_metrics], ignore_index=True))

,model,implementation,train_MAE,test_MAE,train_RMSE,test_RMSE,train_R2,test_R2
0,Linear default,custom,706.024534,717.475660,1023.192388,1040.012607,0.576819,0.589411
1,Ridge default,custom,705.812250,717.316223,1023.172915,1039.981056,0.576835,0.589436
2,Lasso default,custom,705.418130,717.141108,1023.371560,1040.305448,0.576671,0.589180
3,ElasticNet default,custom,799.790866,821.964236,1174.285496,1207.941571,0.442611,0.446112
4,Linear MinMaxScaler,custom,706.024534,717.475660,1023.192388,1040.012607,0.576819,0.589411
5,Ridge MinMaxScaler,custom,705.942845,717.559240,1023.228981,1040.342891,0.576789,0.589150
6,Lasso MinMaxScaler,custom,705.709544,717.643046,1023.607133,1041.195199,0.576476,0.588477
7,ElasticNet MinMaxScaler,custom,1044.550618,1074.706852,1469.546601,1517.860141,0.127073,0.125432
8,Linear StandardScaler,custom,706.024534,717.475660,1023.192388,1040.012607,0.576819,0.589411
9,Ridge StandardScaler,custom,705.815441,717.317790,1023.172906,1039.978415,0.576835,0.589438


## 7. Переобучение на полиномиальных признаках степени 10

In [11]:
polynomial = PolynomialFeatures(degree=10, include_bias=False)
X_train_polynomial = polynomial.fit_transform(train_data[["bathrooms", "bedrooms"]])
X_test_polynomial = polynomial.transform(test_data[["bathrooms", "bedrooms"]])

print(f"Полиномиальных признаков: {X_train_polynomial.shape[1]}")

Полиномиальных признаков: 65


In [12]:
sklearn_polynomial_factories = {
    "Linear Polynomial": lambda: LinearRegression(),
    "Ridge Polynomial": lambda: Ridge(alpha=1.0),
    "Lasso Polynomial": lambda: Lasso(alpha=1.0, max_iter=5000),
    "ElasticNet Polynomial": lambda: ElasticNet(alpha=1.0, l1_ratio=0.5, max_iter=5000),
}

polynomial_rows = []
sklearn_polynomial_models = {}
for model_name, factory in sklearn_polynomial_factories.items():
    model = factory().fit(X_train_polynomial, y_train)
    sklearn_polynomial_models[model_name] = model
    train_metrics = calculate_metrics(y_train, model.predict(X_train_polynomial))
    test_metrics = calculate_metrics(y_test, model.predict(X_test_polynomial))
    polynomial_rows.append({
        "model": model_name,
        "train_MAE": train_metrics["MAE"],
        "test_MAE": test_metrics["MAE"],
        "train_RMSE": train_metrics["RMSE"],
        "test_RMSE": test_metrics["RMSE"],
        "train_R2": train_metrics["R2"],
        "test_R2": test_metrics["R2"],
    })

polynomial_metrics = pd.DataFrame(polynomial_rows)
display(polynomial_metrics)
best_polynomial = polynomial_metrics.loc[polynomial_metrics["test_R2"].idxmax()]
print(
    f"Лучшая из полиномиальных моделей при alpha=1: {best_polynomial['model']} "
    f"(test R2={best_polynomial['test_R2']:.6f})."
)

,model,train_MAE,test_MAE,train_RMSE,test_RMSE,train_R2,test_R2
0,Linear Polynomial,846.825650,865.385796,1203.405424,1259.133564,0.414623,0.398170
1,Ridge Polynomial,751.372511,1061.354163,1069.691396,31371.153181,0.537482,-372.586503
2,Lasso Polynomial,756.633292,769.073277,1077.939837,1099.565247,0.530322,0.541043
3,ElasticNet Polynomial,762.621877,773.543047,1084.294175,1092.439566,0.524768,0.546972


Лучшая из полиномиальных моделей при alpha=1: ElasticNet Polynomial (test R2=0.546972).


In [13]:
custom_polynomial_factories = {
    "Custom Linear Polynomial": lambda: CustomSGDLinearRegression(
        learning_rate=0.01,
        epochs=500,
        batch_size=len(X_train_polynomial),
        random_state=RANDOM_STATE,
    ),
    "Custom Ridge Polynomial": lambda: CustomRidgeRegression(alpha=1.0),
    "Custom Lasso Polynomial": lambda: CustomLassoRegression(
        alpha=1.0, max_iter=150, tol=1e-4
    ),
    "Custom ElasticNet Polynomial": lambda: CustomElasticNetRegression(
        alpha=1.0, l1_ratio=0.5, max_iter=150, tol=1e-4
    ),
}

custom_polynomial_rows = []
for model_name, factory in custom_polynomial_factories.items():
    model = factory().fit(X_train_polynomial, y_train)
    train_metrics = calculate_metrics(y_train, model.predict(X_train_polynomial))
    test_metrics = calculate_metrics(y_test, model.predict(X_test_polynomial))
    custom_polynomial_rows.append({
        "model": model_name,
        "train_RMSE": train_metrics["RMSE"],
        "test_RMSE": test_metrics["RMSE"],
        "train_R2": train_metrics["R2"],
        "test_R2": test_metrics["R2"],
    })

custom_polynomial_metrics = pd.DataFrame(custom_polynomial_rows)
display(custom_polynomial_metrics)

,model,train_RMSE,test_RMSE,train_R2,test_R2
0,Custom Linear Polynomial,1107.008578,1118.010982,0.504649,0.525515
1,Custom Ridge Polynomial,1088.254709,5012.442248,0.521290,-8.537378
2,Custom Lasso Polynomial,1083.108265,1119.888322,0.525807,0.523920
3,Custom ElasticNet Polynomial,1089.089138,1129.866183,0.520556,0.515399


Разрыв между train и test у полиномиальных моделей показывает переобучение: модель высокой степени подстраивается под обучающие сочетания числа комнат и ванных, но нестабильно экстраполирует редкие значения в test. Регуляризация уменьшает этот эффект.

Ниже для Ridge, Lasso и ElasticNet сравниваются несколько значений `alpha`. Лучшим считается значение с максимальным $R^2$ на тестовой части в рамках учебного эксперимента.

In [14]:
alpha_grid = [0.01, 0.1, 1.0, 10.0, 100.0]
alpha_rows = []
for alpha in alpha_grid:
    candidates = {
        "Ridge": Ridge(alpha=alpha),
        "Lasso": Lasso(alpha=alpha, max_iter=1000),
        "ElasticNet": ElasticNet(alpha=alpha, l1_ratio=0.5, max_iter=1000),
    }
    for model_name, model in candidates.items():
        model.fit(X_train_polynomial, y_train)
        train_prediction = model.predict(X_train_polynomial)
        test_prediction = model.predict(X_test_polynomial)
        alpha_rows.append({
            "model": model_name,
            "alpha": alpha,
            "train_R2": r2_custom(y_train, train_prediction),
            "test_R2": r2_custom(y_test, test_prediction),
            "test_RMSE": np.sqrt(mean_squared_error(y_test, test_prediction)),
        })

alpha_results = pd.DataFrame(alpha_rows)
best_alpha_results = alpha_results.loc[alpha_results.groupby("model")["test_R2"].idxmax()]
display(alpha_results)
print("Лучшие alpha:")
display(best_alpha_results.sort_values("test_R2", ascending=False))

,model,alpha,train_R2,test_R2,test_RMSE
0,Ridge,0.01,0.538393,-3023.871823,89266.474499
1,Lasso,0.01,0.529990,0.543994,1096.023770
2,ElasticNet,0.01,0.529063,0.543969,1096.053868
3,Ridge,0.10,0.538222,-1548.522677,63890.143412
4,Lasso,0.10,0.529975,0.543933,1096.097599
5,ElasticNet,0.10,0.527687,0.534314,1107.596289
6,Ridge,1.00,0.537482,-372.586503,31371.153181
7,Lasso,1.00,0.529573,0.543057,1097.150066
8,ElasticNet,1.00,0.522990,0.518364,1126.404360
9,Ridge,10.00,0.536962,-169.924425,21219.588577


Лучшие alpha:


,model,alpha,train_R2,test_R2,test_RMSE
1,Lasso,0.01,0.529990,0.543994,1096.023770
2,ElasticNet,0.01,0.529063,0.543969,1096.053868
12,Ridge,100.00,0.536424,-2.510166,3040.876222


## 8. Наивные модели и итоговые таблицы

In [15]:
naive_rows = []
for model_name, constant in (("Naive mean", y_train.mean()), ("Naive median", np.median(y_train))):
    train_prediction = np.full_like(y_train, constant, dtype=float)
    test_prediction = np.full_like(y_test, constant, dtype=float)
    train_metrics = calculate_metrics(y_train, train_prediction)
    test_metrics = calculate_metrics(y_test, test_prediction)
    naive_rows.append({
        "model": model_name,
        "train_MAE": train_metrics["MAE"],
        "test_MAE": test_metrics["MAE"],
        "train_RMSE": train_metrics["RMSE"],
        "test_RMSE": test_metrics["RMSE"],
        "train_R2": train_metrics["R2"],
        "test_R2": test_metrics["R2"],
    })

final_metrics = pd.concat([
    sklearn_all_metrics.drop(columns="implementation"),
    polynomial_metrics,
    pd.DataFrame(naive_rows),
], ignore_index=True)

result_MAE = final_metrics[["model", "train_MAE", "test_MAE"]].rename(
    columns={"train_MAE": "train", "test_MAE": "test"}
)
result_RMSE = final_metrics[["model", "train_RMSE", "test_RMSE"]].rename(
    columns={"train_RMSE": "train", "test_RMSE": "test"}
)
result_R2 = final_metrics[["model", "train_R2", "test_R2"]].rename(
    columns={"train_R2": "train", "test_R2": "test"}
)

print("MAE")
display(result_MAE)
print("RMSE")
display(result_RMSE)
print("R2")
display(result_R2)

assert len(final_metrics) == 18
assert {"Naive mean", "Naive median"}.issubset(set(final_metrics["model"]))
assert np.isfinite(custom_polynomial_metrics.select_dtypes(include=np.number).to_numpy()).all()
assert polynomial_metrics["test_RMSE"].min() > sklearn_all_metrics["test_RMSE"].min()
assert polynomial_metrics["test_R2"].max() < sklearn_all_metrics["test_R2"].max()
assert polynomial_metrics["test_R2"].min() < 0


MAE


,model,train,test
0,Linear default,705.816155,717.318394
1,Ridge default,705.812250,717.316223
2,Lasso default,705.419664,717.142644
3,ElasticNet default,799.790808,821.964181
4,Linear MinMaxScaler,705.816155,717.318394
5,Ridge MinMaxScaler,705.942845,717.559240
6,Lasso MinMaxScaler,705.710307,717.643855
7,ElasticNet MinMaxScaler,1044.550721,1074.706996
8,Linear StandardScaler,705.816155,717.318394
9,Ridge StandardScaler,705.815441,717.317790


RMSE


,model,train,test
0,Linear default,1023.172906,1039.977925
1,Ridge default,1023.172915,1039.981056
2,Lasso default,1023.372627,1040.306704
3,ElasticNet default,1174.285421,1207.941502
4,Linear MinMaxScaler,1023.172906,1039.977925
5,Ridge MinMaxScaler,1023.228981,1040.342891
6,Lasso MinMaxScaler,1023.607668,1041.195805
7,ElasticNet MinMaxScaler,1469.546759,1517.860333
8,Linear StandardScaler,1023.172906,1039.977925
9,Ridge StandardScaler,1023.172906,1039.978415


R2


,model,train,test
0,Linear default,0.576835,0.589438
1,Ridge default,0.576835,0.589436
2,Lasso default,0.576670,0.589179
3,ElasticNet default,0.442611,0.446112
4,Linear MinMaxScaler,0.576835,0.589438
5,Ridge MinMaxScaler,0.576789,0.589150
6,Lasso MinMaxScaler,0.576476,0.588476
7,ElasticNet MinMaxScaler,0.127072,0.125431
8,Linear StandardScaler,0.576835,0.589438
9,Ridge StandardScaler,0.576835,0.589438


In [16]:
regular_candidates = final_metrics[
    (~final_metrics["model"].str.contains("Polynomial"))
    & (~final_metrics["model"].str.contains("Naive"))
].copy()

regular_candidates["R2_gap"] = (
    regular_candidates["train_R2"] - regular_candidates["test_R2"]
).abs()
best_model = regular_candidates.loc[regular_candidates["test_R2"].idxmax()]

# Различия между ведущими линейными моделями малы и зависят от split/sklearn.
# Среди моделей в пределах 0.002 R2 от лучшей выбираем наиболее стабильную.
near_best = regular_candidates[
    regular_candidates["test_R2"] >= best_model["test_R2"] - 0.002
]
most_stable_model = near_best.sort_values(
    ["R2_gap", "test_RMSE"], ascending=[True, True]
).iloc[0]

print(
    f"Лучшая модель строго по test R2: {best_model['model']} "
    f"(R2={best_model['test_R2']:.6f}, MAE={best_model['test_MAE']:.3f}, "
    f"RMSE={best_model['test_RMSE']:.3f})."
)
print(
    f"Наиболее стабильная среди практически равных лидеров: {most_stable_model['model']} "
    f"(|train R2 - test R2|={most_stable_model['R2_gap']:.6f})."
)


Лучшая модель строго по test R2: Linear default (R2=0.589438, MAE=717.318, RMSE=1039.978).
Наиболее стабильная среди практически равных лидеров: Lasso MinMaxScaler (|train R2 - test R2|=0.012001).


## 9. Дополнительное задание

### 9.1. Преобразование целевой переменной

У `price` тяжёлый правый хвост. Сравним Ridge на исходной цели и на $\log(price)$ с обязательным обратным преобразованием $\exp(\hat y_{log})$. Чтобы линейная модель могла описывать умеренную нелинейность, переиспользуем описанный в теории подход $\phi(x)$: полиномиальные взаимодействия степени 2. Границы обработки числовых признаков и scaler оцениваются только по train.

In [17]:
def prepare_bonus_features(train_frame, test_frame):
    X_tr = train_frame[feature_list].to_numpy(dtype=float).copy()
    X_te = test_frame[feature_list].to_numpy(dtype=float).copy()

    # Пределы определяются исключительно по train и применяются к обеим частям.
    numeric_low = np.quantile(X_tr[:, -2:], 0.001, axis=0)
    numeric_high = np.quantile(X_tr[:, -2:], 0.999, axis=0)
    X_tr[:, -2:] = np.clip(X_tr[:, -2:], numeric_low, numeric_high)
    X_te[:, -2:] = np.clip(X_te[:, -2:], numeric_low, numeric_high)

    transformer = PolynomialFeatures(degree=2, include_bias=False)
    X_tr_poly = transformer.fit_transform(X_tr)
    X_te_poly = transformer.transform(X_te)
    scaler = StandardScaler()
    return scaler.fit_transform(X_tr_poly), scaler.transform(X_te_poly)


X_bonus_train, X_bonus_test = prepare_bonus_features(train_data, test_data)

bonus_raw_target = Ridge(alpha=1.0).fit(X_bonus_train, y_train)
bonus_log_target = Ridge(alpha=1.0).fit(X_bonus_train, np.log(y_train))

bonus_raw_prediction = np.clip(
    bonus_raw_target.predict(X_bonus_test), y_train.min(), y_train.max()
)
bonus_log_prediction = np.clip(
    np.exp(bonus_log_target.predict(X_bonus_test)), y_train.min(), y_train.max()
)

target_transform_results = pd.DataFrame([
    {
        "target": "price",
        **calculate_metrics(y_test, bonus_raw_prediction),
    },
    {
        "target": "log(price), inverse exp",
        **calculate_metrics(y_test, bonus_log_prediction),
    },
])
display(target_transform_results)
print("Skew price:", train_data["price"].skew())
print("Skew log(price):", np.log(train_data["price"]).skew())
assert target_transform_results.loc[1, "R2"] > 0.61

,target,MAE,RMSE,R2
0,price,671.251265,973.576431,0.640192
1,"log(price), inverse exp",661.589486,989.392509,0.628407


Skew price: 2.001549924524958
Skew log(price): 0.59020039591573


Логарифмирование уменьшает асимметрию цели. После прогноза обязательно выполняется обратное преобразование, а результаты сравниваются в исходных денежных единицах.

### 9.2. Выбросы только в обучающей выборке

Основная выборка уже прошла обязательное отсечение крайних 1% до разделения, как в ML1. В дополнительном эксперименте границы следующего 1-го и 99-го процентилей вычисляются только по `bonus_train_raw`, и дополнительное удаление выполняется только в этой обучающей части. Зафиксированный `bonus_test_raw` не меняется, поскольку очистка контрольной выборки после разделения искусственно улучшила бы оценку.


In [18]:
def fit_bonus_ridge(train_frame, test_frame):
    X_tr, X_te = prepare_bonus_features(train_frame, test_frame)
    y_tr = train_frame["price"].to_numpy(dtype=float)
    model = Ridge(alpha=1.0).fit(X_tr, y_tr)
    prediction = np.clip(model.predict(X_te), y_tr.min(), y_tr.max())
    return calculate_metrics(test_frame["price"].to_numpy(dtype=float), prediction)


test_rows_before_bonus = len(bonus_test_raw)
outlier_results = pd.DataFrame([
    {
        "training_data": "train with price outliers",
        **fit_bonus_ridge(bonus_train_raw, bonus_test_raw),
    },
    {
        "training_data": "train without price outliers",
        **fit_bonus_ridge(bonus_train_clean, bonus_test_raw),
    },
])
display(outlier_results)
print("Строк test до и после эксперимента:", test_rows_before_bonus, len(bonus_test_raw))
assert len(bonus_test_raw) == test_rows_before_bonus
assert outlier_results.loc[1, "R2"] > 0.61


,training_data,MAE,RMSE,R2
0,train with price outliers,671.251265,973.576431,0.640192
1,train without price outliers,673.268178,987.722429,0.629660


Строк test до и после эксперимента: 12086 12086


### 9.3. Аналитическое решение, batch GD и mini-batch SGD

Аналитическая реализация использует псевдообратную матрицу. Batch GD вычисляет градиент по всей обучающей выборке, а mini-batch SGD — по воспроизводимо перемешиваемым пакетам. Все варианты имеют методы `fit` и `predict` и сравниваются со `sklearn` на одинаковых данных.

In [19]:
class CustomAnalyticalLinearRegression:
    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)
        design = np.column_stack([np.ones(len(X)), X])
        theta = np.linalg.pinv(design) @ y
        self.intercept_ = theta[0]
        self.coef_ = theta[1:]
        return self

    def predict(self, X):
        return np.asarray(X, dtype=float) @ self.coef_ + self.intercept_


bonus_algorithm_models = {
    "Analytical": CustomAnalyticalLinearRegression(),
    "Batch GD": CustomSGDLinearRegression(
        learning_rate=0.02, epochs=500, batch_size=len(X_train), random_state=RANDOM_STATE
    ),
    "Mini-batch SGD": CustomSGDLinearRegression(
        learning_rate=0.02, epochs=250, batch_size=256, random_state=RANDOM_STATE
    ),
    "sklearn": LinearRegression(),
}

algorithm_rows = []
for model_name, model in bonus_algorithm_models.items():
    model.fit(X_train, y_train)
    prediction = model.predict(X_test)
    algorithm_rows.append({"model": model_name, **calculate_metrics(y_test, prediction)})

algorithm_comparison = pd.DataFrame(algorithm_rows)
display(algorithm_comparison)
sklearn_rmse = algorithm_comparison.loc[algorithm_comparison["model"] == "sklearn", "RMSE"].iloc[0]
assert np.all(
    np.abs(algorithm_comparison.loc[:2, "RMSE"] - sklearn_rmse) / sklearn_rmse < 0.10
)

,model,MAE,RMSE,R2
0,Analytical,717.318394,1039.977925,0.589438
1,Batch GD,716.489087,1040.746141,0.588832
2,Mini-batch SGD,717.475660,1040.012607,0.589411
3,sklearn,717.318394,1039.977925,0.589438


## 10. Вывод

Собственные реализации сопоставлены со `sklearn` на одних и тех же 22 признаках. Единый очищенный набор из ML1 разделён на `train` и `test`; исходный Kaggle `test.json` не подменяет контрольную часть. Обе собственные нормализации численно совпадают со `sklearn`, а scaler обучается только на `train`. Полиномы степени 10 на двух исходных признаках демонстрируют переобучение, регуляризация уменьшает его. Среднее и медиана добавлены как наивные базовые прогнозы.

В дополнительной части проверены преобразование цели, удаление выбросов только из отдельной обучающей части при неизменном test, аналитическое решение, batch GD и mini-batch SGD. Контрольные $R^2$ для пунктов 11.a и 11.b должны превышать 0.61, а реализации пункта 11.c сравниваются со `sklearn` на одной выборке.
